# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas  
**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI).

**Nota de elaboración:** Las celdas Markdown cuentan con apoyo de inteligencia artificial generativa. Los ejemplos tienen fines educativos y demostrativos. El autor es responsable de su revisión, adaptación y validación en Microsoft Fabric.

# NB_03 — Análisis de sentimientos con `ai.analyze_sentiment`

**Serie:** Microsoft Fabric AI Functions con PySpark  
**Modelo:** `gpt-5-mini`

## Objetivos

- Clasificar opiniones de clientes como `positive`, `negative`, `mixed` o `neutral`.
- Probar etiquetas personalizadas y revisar errores.
- Medir tokens, estimar CU-segundos y registrar auditoría por ejecución.

## Documentación oficial

- [Analyze sentiment](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/analyze-sentiment)
- [Configuración PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)

## 1. Requisitos

Capacidad Fabric compatible, AI Functions habilitadas por la administración y sesión PySpark con SynapseML. Para Runtime 2.0 (Spark 4.1), revisar la configuración temporal indicada en Microsoft Learn. Adjuntar un Lakehouse solo si se desea persistir. Usar datos ficticios: el texto se procesa mediante el endpoint del modelo. Cada llamada puede consumir capacidad.

## 2. Configuración del modelo

Configuramos el deployment generativo de AI Functions para la sesión actual. El modelo configurado debe contrastarse con el modelo reportado en las estadísticas. Las llamadas pueden generar consumo real de capacidad.

In [ ]:
from synapse.ml.spark import aifunc
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import uuid

MODEL_NAME = "gpt-5-mini"
aifunc.default_conf.set_deployment_name(MODEL_NAME)
print(aifunc.default_conf.get_deployment_name())

## 3. Referencia del método publicado por Microsoft

[Use ai.analyze_sentiment with PySpark — Microsoft Learn](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/analyze-sentiment)

### Sintaxis oficial (referencia, no ejecutable)

```python
# Categorías estándar
df.ai.analyze_sentiment(input_col="input", output_col="sentiment")

# Categorías personalizadas
df.ai.analyze_sentiment(input_col="input", output_col="sentiment", labels=["happy", "angry", "indifferent"])
```

### Parámetros

| Parámetro | Tipo | Requerido | Comportamiento si se omite |
|---|---|---|---|
| `input_col` | str | Sí | No aplica |
| `output_col` | str | No | Nombre generado automáticamente |
| `labels` | secuencia de str | No | positive, negative, mixed, neutral |
| `error_col` | str | No | Nombre generado automáticamente |

**Retorno:** un nuevo DataFrame con una etiqueta por fila; `null` cuando no se determina el sentimiento. La columna de errores es nula si no hay error. Para archivos compatibles se documenta también `input_col_type="path"`.

Los ejemplos siguientes son propios de esta serie, con opiniones ficticias de clientes. No son resultados publicados por Microsoft.

| Etiqueta | Interpretación didáctica |
|---|---|
| positive | valoración favorable |
| negative | valoración desfavorable |
| mixed | combina aspectos favorables y desfavorables |
| neutral | texto sin valoración clara, por ejemplo un hecho descriptivo |

No interpretar una etiqueta como una probabilidad ni como una evaluación clínica.


## 4. Dataset ficticio

Las etiquetas esperadas son referencias humanas para evaluar el ejemplo, no resultados garantizados.

In [ ]:
data = [
    (1, "Excelente atención, resolvieron mi problema rápidamente.", "positive"),
    (2, "El servicio fue pésimo y nadie respondió mi reclamo.", "negative"),
    (3, "El producto es excelente, pero la entrega fue terrible.", "mixed"),
    (4, "El pedido llegó el martes a las 10 de la mañana.", "neutral"),
    (5, "Me encantó la facilidad de uso.", "positive"),
    (6, "La aplicación falla constantemente, no la recomiendo.", "negative"),
    (7, "Buen precio, aunque la calidad me decepcionó.", "mixed"),
    (8, "La solicitud tiene el número 1234.", "neutral")
]
df = spark.createDataFrame(data, ["id_opinion", "opinion", "sentimiento_esperado"])
display(df)

## 5. Análisis con etiquetas predeterminadas

`input_col` identifica el texto, `output_col` la nueva columna y `error_col` los errores. Una opinión sin sentimiento determinable puede devolver `null`. No confundir neutral con resultado indeterminado.

Materializamos el resultado en caché para reducir reevaluaciones; Spark puede recomputar particiones si pierde la caché.

In [ ]:
run_id_default = str(uuid.uuid4())
resultado = df.ai.analyze_sentiment(
    input_col="opinion", output_col="sentimiento", error_col="error_sentimiento"
)
resultado.cache()
resultado.count()
display(resultado)

## 6. Validar resultados y distribución

Comparar la etiqueta obtenida con la referencia humana y revisar por separado los casos nulos o con errores. La coincidencia de ocho ejemplos no constituye una evaluación estadística del modelo.

In [ ]:
resultado.printSchema()
display(resultado.groupBy("sentimiento").count())
display(resultado.withColumn("coincide", F.coalesce(F.col("sentimiento") == F.col("sentimiento_esperado"), F.lit(False))))
display(resultado.filter(F.col("error_sentimiento").isNotNull() | F.col("sentimiento").isNull()))

## 7. Consultar estadísticas con `ai.stats`

Una vez materializado el resultado, consultamos el DataFrame de estadísticas. Inspeccionar el esquema y la granularidad antes de interpretar los datos. Cada llamada se registra de forma independiente.

| Métrica | Interpretación |
|---|---|
| `input_tokens` | Entrada total |
| `cached_tokens` | Subconjunto de entrada servido desde caché |
| `output_tokens` | Salida total |
| `reasoning_tokens` | Razonamiento, informativo; no sumar nuevamente a salida si ya está incluido |
| `num_successful` | Filas procesadas exitosamente |
| `num_exceptions` | Excepciones reportadas |

[Configuración y estadísticas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration).

In [ ]:
def read_stats(result):
    stats_df = result.ai.stats
    display(stats_df)
    rows = stats_df.collect()
    if len(rows) != 1:
        raise RuntimeError("Revisar el esquema y la granularidad de ai.stats antes de estimar CU.")
    return rows[0].asDict(recursive=True)

stats_default = read_stats(resultado)
for key in ("input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens"):
    print(f"{key}: {stats_default.get(key)}")

## 8. Estimar el consumo de capacidad

Tarifas de referencia de la serie para `gpt-5-mini` integrado:

| Tipo de token | CU-s por 1.000 tokens |
|---|---:|
| Entrada sin caché | 8,40 |
| Entrada en caché | 0,84 |
| Salida | 67,23 |

$$CU_s = \frac{(I-C)\times8.40+C\times0.84+O\times67.23}{1000}$$

I: entrada total; C: entrada en caché; O: salida. Ejemplo ilustrativo: 1.000 tokens de entrada, 200 en caché y 100 de salida equivalen a 13,611 CU-s. No incluye Spark. Verificar modelo y tarifas vigentes en [Microsoft Learn — Billing](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing). Si faltan métricas, el código detiene el cálculo en lugar de inventar valores.

In [ ]:
def estimate_cu(stats):
    i, c, o = (stats.get(k) for k in ("input_tokens", "cached_tokens", "output_tokens"))
    if any(v is None for v in (i, c, o)):
        raise ValueError("Métricas de tokens incompletas; no es posible estimar CU.")
    i, c, o = int(i), int(c), int(o)
    if not 0 <= c <= i or o < 0:
        raise ValueError("Métricas inconsistentes")
    return ((i-c)*8.40 + c*0.84 + o*67.23)/1000

print("CU-segundos estimados:", estimate_cu(stats_default))

## 9. Etiquetas personalizadas

Es posible adaptar las categorías al negocio mediante `labels`. Esta es otra inferencia, con consumo independiente. Las etiquetas en español elegidas son satisfecho, insatisfecho, ambivalente y sin opinión.

In [ ]:
run_id_custom = str(uuid.uuid4())
resultado_custom = df.ai.analyze_sentiment(
    input_col="opinion", output_col="sentimiento_personalizado",
    labels=["satisfecho", "insatisfecho", "ambivalente", "sin opinión"],
    error_col="error_personalizado"
)
resultado_custom.cache()
resultado_custom.count()
display(resultado_custom)
stats_custom = read_stats(resultado_custom)
print("CU-segundos estimados:", estimate_cu(stats_custom))

## 10. Auditoría de ambas llamadas

Se registra una fila por ejecución, sin almacenar los textos originales. Los campos ausentes permanecen nulos. El estado refleja resultados y errores observados, no garantiza exactitud semántica.

In [ ]:
audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])


def audit_row(run_id, stats, result, output, error):
    total = df.count()
    invalid = result.filter(F.col(output).isNull() | F.col(error).isNotNull()).count()
    def metric(k):
        return int(stats[k]) if stats.get(k) is not None else None
    return (run_id, datetime.now(timezone.utc).isoformat(),
            "NB_03_AI_ANALYZE_SENTIMENT", "ai.analyze_sentiment", MODEL_NAME,
            str(stats["model"]) if stats.get("model") is not None else None,
            total, metric("num_successful"), metric("num_exceptions"),
            metric("num_unevaluated"), metric("num_harmful"),
            metric("input_tokens"), metric("cached_tokens"), metric("output_tokens"),
            metric("reasoning_tokens"), float(estimate_cu(stats)),
            str(stats["client_type"]) if stats.get("client_type") is not None else None,
            "SUCCESS" if invalid == 0 else "PARTIAL" if invalid < total else "INCOMPLETE")

audit_df = spark.createDataFrame([
    audit_row(run_id_default, stats_default, resultado, "sentimiento", "error_sentimiento"),
    audit_row(run_id_custom, stats_custom, resultado_custom, "sentimiento_personalizado", "error_personalizado")
], audit_schema)
display(audit_df)
display(audit_df.agg(F.sum("estimated_cu_seconds").alias("total_cu_seconds")))

## 11. Persistencia opcional en Delta

Se mantiene el esquema de auditoría de la serie. Activar únicamente con un Lakehouse adjunto y ajustar el nombre si no utiliza schemas. Reejecutar la escritura en modo append puede duplicar registros.

In [ ]:
PERSIST_AUDIT = False
TABLE_NAME = "dbo.audit_ai_functions"
if PERSIST_AUDIT:
    audit_df.write.format("delta").mode("append").saveAsTable(TABLE_NAME)

## 12. Liberar caché

In [ ]:
resultado.unpersist()
resultado_custom.unpersist()

## 13. Conclusiones

La función permite clasificar emociones con categorías estándar o personalizadas. Revisar casos ambiguos y errores antes de automatizar decisiones. Contrastar las estimaciones con Fabric Capacity Metrics y validar el comportamiento en el runtime real.

**Siguiente notebook:** NB_04 — AI_SUMMARIZE.